In [ ]:
from langgraph.graph import StateGraph,START,END
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langchain_core.messages import HumanMessage,SystemMessage,BaseMessage
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

In [ ]:
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages] # here we can also use operator.add as reducer, but add_messages is especially optimized for chatbots and is recommended by langgraph.


In [ ]:
def chat_node(state:ChatState) -> ChatState:
    messages = state['messages']
    res = model.invoke(messages)
    return {"messages":[res]}

In [ ]:
checkpointer = InMemorySaver()

graph = StateGraph(ChatState)

graph.add_node("chat_node",chat_node)

graph.add_edge(START,"chat_node")
graph.add_edge("chat_node",END)

chatbot = graph.compile(checkpointer=checkpointer)


In [ ]:
initial_state = {"messages": [HumanMessage("Who is PM of Pakistan?"),SystemMessage("You are a hindi standup comedian. reply in funny way, in hindi but english text.")]}
chatbot.invoke(initial_state)["messages"][-1].content

In [ ]:
# streaming the response
config = {'configurable':{"thread_id":"thread_1"}}
for message_chunk, metadata in chatbot.stream({"messages": [HumanMessage("Write 500 word essay on global warming.")]},config=config,stream_mode="messages"):
    if message_chunk.content:
        print(message_chunk.content,end=" ",flush=True)

In [ ]:
thread_id = '1'

while True:

    user_message = input("Write your message here: ")
    print(user_message)

    if user_message.strip().lower() in ["exit","quit"]:
        break

    config = {'configurable':{"thread_id":thread_id}}
    response = chatbot.invoke({"messages":[HumanMessage(user_message)]},config=config)
    print(f"AI: {response['messages'][-1].content}")

In [ ]:
chatbot.get_state(config)

In [ ]:
list(chatbot.get_state_history(config))